In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from libpysal import graph
import geopandas as gpd
from scipy.optimize import linear_sum_assignment
from matplotlib.colors import ListedColormap
from matplotlib.ticker import AutoMinorLocator
from matplotlib.ticker import MultipleLocator
import tools
import colorsys
import matplotlib.colors as mcolors

# Synthetic

In [ ]:
sizes = [100,400]
shapes = ["hex","pent","square"]

In [ ]:
dfs = []
for shape in shapes:
    for size in sizes:
        path = f"summary/regionalization/synthetic/metrics_{shape}_{size}.parquet"
        df = pd.read_parquet(path)
        df["shape"] = shape
        df["size"] = size
        dfs.append(df)

gdf = pd.concat(dfs, ignore_index=True)

In [ ]:
total_edges = {}
for shape in shapes:
    for size in sizes:
        path = f"graphs/{shape}/size_{size}/g_true.parquet"
        g = graph.read_parquet(path)
        total_edges[(shape, size)] = g.n_edges / 2  # undirected: adjacency table has both (i,j) and (j,i)

gdf['total_edges'] = gdf.apply(lambda row: total_edges[(row['shape'], row['size'])], axis=1)
gdf['pct_missing'] = gdf['missing_edges'] / gdf['total_edges'] * 100

In [ ]:
gdf

In [ ]:
# rows where either algorithm has ARI <= 0
gdf[(gdf['ari_ward'] <= 0) | (gdf['ari_skater'] <= 0)]

In [ ]:
subset = gdf.query("method == 'random'").copy()
subset['layout'] = subset['shape'] + '_' + subset['size'].astype(str)
subset['pct_missing'] = subset['pct_missing'].round(1)

# --- bin into 5% intervals ---
max_pct = subset['pct_missing'].max()
bin_edges = np.arange(0, np.ceil(max_pct / 5) * 5 + 5, 5)
subset['missing_bin'] = pd.cut(subset['pct_missing'], bins=bin_edges,
                                right=False, include_lowest=True)

def fmt_bin(iv):
    return f"{int(iv.left)}-{int(iv.right)}"

subset['missing_bin_label'] = subset['missing_bin'].map(fmt_bin)
bin_order = [fmt_bin(iv) for iv in subset['missing_bin'].cat.categories]

long_df = subset.melt(
    id_vars=['layout', 'missing_bin_label'],
    value_vars=['ari_ward', 'ari_skater'],
    var_name='algorithm', value_name='ari'
)
long_df['algorithm'] = long_df['algorithm'].map({'ari_ward': 'SCHC', 'ari_skater': 'SKATER'})

layouts = subset['layout'].unique()

from matplotlib.ticker import MultipleLocator

fig, axes = plt.subplots(3, 2, figsize=(8.27, 11.0), sharex=True, sharey=True)
for i, (ax, layout) in enumerate(zip(axes.flat, layouts)):
    d = long_df[long_df['layout'] == layout]
    sns.boxplot(data=d, x='missing_bin_label', y='ari', hue='algorithm', ax=ax,
                order=bin_order, flierprops={'markersize': 3})
    ax.set_title(layout, fontsize=10)
    ax.tick_params(axis='both', labelsize=8)
    ax.tick_params(axis='x', rotation=90)
    ax.legend(fontsize=7, loc='lower left')

    # --- finer gridlines ---
    ax.yaxis.set_major_locator(MultipleLocator(0.2))   # major gridlines every 0.2
    ax.yaxis.set_minor_locator(MultipleLocator(0.05))  # minor gridlines every 0.05
    ax.grid(which='major', alpha=0.35)
    ax.grid(which='minor', alpha=0.15, linestyle=':')

    row, col = divmod(i, 2)
    ax.set_xlabel('% edges missing' if row == 2 else '', fontsize=9)
    ax.set_ylabel('ARI' if col == 0 else '', fontsize=9)
    if col != 0:
        ax.tick_params(axis='y', labelleft=False)
sns.despine()
fig.tight_layout(rect=[0.02, 0.01, 0.98, 0.965])
plt.savefig("figs/reg_boxplots.png", dpi=600, bbox_inches="tight")

In [ ]:
subset = gdf.query("method == 'random'").copy()
subset['layout'] = subset['shape'] + '_' + subset['size'].astype(str)
subset['pct_missing'] = subset['pct_missing'].round(1)

# --- bin into 2.5% intervals ---
bin_width = 5
max_pct = subset['pct_missing'].max()
bin_edges = np.arange(0, np.ceil(max_pct / bin_width) * bin_width + bin_width, bin_width)
subset['missing_bin'] = pd.cut(subset['pct_missing'], bins=bin_edges,
                                right=False, include_lowest=True)

def fmt_bin(iv):
    return f"{iv.left:g}-{iv.right:g}"

subset['missing_bin_label'] = subset['missing_bin'].map(fmt_bin)
bin_order = [fmt_bin(iv) for iv in subset['missing_bin'].cat.categories]

long_df = subset.melt(
    id_vars=['layout', 'missing_bin_label'],
    value_vars=['ari_ward', 'ari_skater'],
    var_name='algorithm', value_name='ari'
)
long_df['algorithm'] = long_df['algorithm'].map({'ari_ward': 'SCHC', 'ari_skater': 'SKATER'})
long_df['missing_bin_label'] = pd.Categorical(
    long_df['missing_bin_label'], categories=bin_order, ordered=True
)

layouts = subset['layout'].unique()

fig, axes = plt.subplots(3, 2, figsize=(8.27, 11.0), sharex=True, sharey=True)

for i, (ax, layout) in enumerate(zip(axes.flat, layouts)):
    d = long_df[long_df['layout'] == layout]
    sns.stripplot(
        data=d, x='missing_bin_label', y='ari', hue='algorithm',
        order=bin_order, dodge=True, jitter=0.25,
        size=2.2, alpha=0.4, linewidth=0, ax=ax
    )
    ax.set_title(layout, fontsize=10)
    ax.tick_params(axis='both', labelsize=8)
    ax.tick_params(axis='x', rotation=90, labelsize=6)

    # --- finer gridlines ---
    ax.yaxis.set_major_locator(MultipleLocator(0.1))
    ax.yaxis.set_minor_locator(MultipleLocator(0.025))
    ax.grid(which='major', axis='y', alpha=0.35)
    ax.grid(which='minor', axis='y', alpha=0.15, linestyle=':')
    ax.grid(which='major', axis='x', alpha=0.15)

    # --- reference lines ---
    ax.axhline(0, color='black', lw=0.9, linestyle='--', alpha=0.6)
    ax.axhline(1, color='black', lw=0.9, linestyle='--', alpha=0.6)

    row, col = divmod(i, 2)
    ax.set_xlabel('% edges missing' if row == 2 else '', fontsize=9)
    ax.set_ylabel('ARI' if col == 0 else '', fontsize=9)
    if col != 0:
        ax.tick_params(axis='y', labelleft=False)

    # --- legend only on first panel ---
    if i == 0:
        ax.legend(fontsize=7, loc='lower left', title='Algorithm', title_fontsize=7)
    else:
        if ax.get_legend() is not None:
            ax.get_legend().remove()

fig.tight_layout(rect=[0.02, 0.01, 0.98, 0.965])
plt.savefig("figs/reg_scatterplots.png", dpi=600, bbox_inches="tight")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import colorsys
import matplotlib.colors as mcolors
from matplotlib.ticker import AutoMinorLocator

# --- prep ---
subset = gdf.query("method == 'random'").copy()
subset['layout'] = subset['shape'] + '_' + subset['size'].astype(str)
subset['pct_missing'] = subset['pct_missing'].round(1)

# --- bin into 5% intervals ---
bin_width = 5
max_pct = subset['pct_missing'].max()
bin_edges = np.arange(0, np.ceil(max_pct / bin_width) * bin_width + bin_width, bin_width)
subset['missing_bin'] = pd.cut(subset['pct_missing'], bins=bin_edges,
                                right=False, include_lowest=True)

def fmt_bin(iv):
    return f"{iv.left:g}-{iv.right:g}"

subset['missing_bin_label'] = subset['missing_bin'].map(fmt_bin)
bin_order = [fmt_bin(iv) for iv in subset['missing_bin'].cat.categories]

long_df = subset.melt(
    id_vars=['layout', 'missing_bin_label'],
    value_vars=['ari_ward', 'ari_skater'],
    var_name='algorithm', value_name='ari'
)
long_df['algorithm'] = long_df['algorithm'].map({'ari_ward': 'SCHC', 'ari_skater': 'SKATER'})

# --- aggregate: mean ARI per layout, algorithm, bin ---
mean_df = (
    long_df.groupby(['layout', 'algorithm', 'missing_bin_label'], observed=True)['ari']
    .mean()
    .reset_index()
)

mean_df['missing_bin_label'] = pd.Categorical(
    mean_df['missing_bin_label'], categories=bin_order, ordered=True
)
mean_df = mean_df.sort_values(['layout', 'algorithm', 'missing_bin_label'])

# --- color / marker maps: base hue per shape, shade per size ---
def adjust_lightness(hex_color, factor):
    r, g, b = mcolors.to_rgb(hex_color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = max(0, min(1, l * factor))
    return colorsys.hls_to_rgb(h, l, s)

base_colors = {
    'hex': '#1f77b4',
    'pent': '#ff7f0e',
    'square': '#2ca02c',
}

color_map = {}
marker_map = {}
for shape, base in base_colors.items():
    color_map[f'{shape}_100'] = adjust_lightness(base, 1.5)
    color_map[f'{shape}_400'] = adjust_lightness(base, 0.75)
    marker_map[f'{shape}_100'] = 'o'
    marker_map[f'{shape}_400'] = 's'

# --- plot ---
fig, axes = plt.subplots(1, 2, figsize=(11, 6), sharey=True)
for ax, algo in zip(axes, ['SCHC', 'SKATER']):
    d = mean_df[mean_df['algorithm'] == algo].sort_values(['layout', 'missing_bin_label'])
    sns.lineplot(
        data=d, x='missing_bin_label', y='ari',
        hue='layout', style='layout',
        palette=color_map, markers=marker_map, dashes=False,
        markersize=6, linewidth=1.5, ax=ax
    )
    ax.set_title(algo, fontsize=11)
    ax.set_xlabel('% edges missing', fontsize=10)
    ax.set_ylabel('Mean ARI' if algo == 'SCHC' else '', fontsize=10)
    ax.tick_params(axis='x', rotation=90, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    # fine grid on y only (major + minor)
    ax.yaxis.set_minor_locator(AutoMinorLocator(4))
    ax.grid(which='major', axis='y', alpha=0.3, linewidth=0.6)
    ax.grid(which='minor', axis='y', alpha=0.15, linewidth=0.4, linestyle=':')
    # x: gridline only at each category tick, no finer subdivision
    ax.grid(which='major', axis='x', alpha=0.3, linewidth=0.6)
    if algo == 'SCHC':
        ax.legend(fontsize=7, loc='lower left')
    else:
        ax.legend_.remove()
sns.despine()
fig.tight_layout()
plt.savefig("figs/reg_lineplot_faceted.png", dpi=600, bbox_inches="tight")

In [ ]:
# aggregate: minimum ARI per layout, algorithm, bin
min_df = (
    long_df.groupby(['layout', 'algorithm', 'missing_bin_label'], observed=True)['ari']
    .max()
    .reset_index()
)
min_df['missing_bin_label'] = pd.Categorical(
    min_df['missing_bin_label'], categories=bin_order, ordered=True
)
min_df['layout'] = pd.Categorical(min_df['layout'], categories=layout_order, ordered=True)
min_df = min_df.sort_values(['layout', 'algorithm', 'missing_bin_label'])

fig, axes = plt.subplots(1, 2, figsize=(11, 6), sharey=True)
for ax, algo in zip(axes, ['SCHC', 'SKATER']):
    d = min_df[min_df['algorithm'] == algo].sort_values(['layout', 'missing_bin_label'])
    sns.lineplot(
        data=d, x='missing_bin_label', y='ari',
        hue='layout', style='layout',
        palette=color_map, markers=marker_map, dashes=False,
        markersize=6, linewidth=1.5, ax=ax
    )
    ax.set_title(algo, fontsize=11)
    ax.set_xlabel('% edges missing', fontsize=10)
    ax.set_ylabel('Min ARI' if algo == 'SCHC' else '', fontsize=10)
    ax.tick_params(axis='x', rotation=90, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    # fine grid on y only (major + minor)
    ax.yaxis.set_minor_locator(AutoMinorLocator(4))
    ax.grid(which='major', axis='y', alpha=0.3, linewidth=0.6)
    ax.grid(which='minor', axis='y', alpha=0.15, linewidth=0.4, linestyle=':')
    # x: gridline only at each category tick, no finer subdivision
    ax.grid(which='major', axis='x', alpha=0.3, linewidth=0.6)
    if algo == 'SCHC':
        ax.legend(fontsize=7, loc='lower left')
    else:
        ax.legend_.remove()
fig.tight_layout()
sns.despine()
#plt.savefig("figs/reg_lineplot_faceted_min.png", dpi=600, bbox_inches="tight")

In [ ]:
min_mean_ari = (
    mean_df.groupby(['layout', 'algorithm'], observed=True)['ari']
    .min()
    .reset_index()
    .rename(columns={'ari': 'min_mean_ari'})
    .sort_values(['algorithm', 'layout'])
)

print(min_mean_ari.to_string(index=False))


In [ ]:
subset = gdf.copy()
mask_drop = subset['method'].isin(['borders', 'local']) & (subset['c_run'] == subset['z_run'])
subset = subset[~mask_drop]
subset['layout'] = subset['shape'] + '_' + subset['size'].astype(str)

# --- bin into 5% intervals ---
bin_width = 5
max_pct = subset['pct_missing'].max()
bin_edges = np.arange(0, np.ceil(max_pct / bin_width) * bin_width + bin_width, bin_width)
subset['missing_bin'] = pd.cut(subset['pct_missing'], bins=bin_edges,
                                right=False, include_lowest=True)

def fmt_bin(iv):
    return f"{iv.left:g}-{iv.right:g}"

subset['missing_bin_label'] = subset['missing_bin'].map(fmt_bin)
bin_order = [fmt_bin(iv) for iv in subset['missing_bin'].cat.categories]

long_df = subset.melt(
    id_vars=['layout', 'method', 'missing_bin_label'],
    value_vars=['ari_ward', 'ari_skater'],
    var_name='algorithm', value_name='ari'
)
long_df['algorithm'] = long_df['algorithm'].map({'ari_ward': 'Ward', 'ari_skater': 'SKATER'})
long_df['missing_bin_label'] = pd.Categorical(
    long_df['missing_bin_label'], categories=bin_order, ordered=True
)

grouped = (
    long_df.groupby(['layout', 'method', 'algorithm', 'missing_bin_label'], observed=True)['ari']
    .mean()
    .reset_index()
)

layouts = subset['layout'].unique()
methods = subset['method'].unique()

# --- base color per method (blue / orange / green family), shaded by algorithm ---
def adjust_lightness(hex_color, factor):
    r, g, b = mcolors.to_rgb(hex_color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = max(0, min(1, l * factor))
    return colorsys.hls_to_rgb(h, l, s)

base_colors = {
    'random':  '#1f77b4',  # blue
    'borders': '#2ca02c',  # green
    'local':   '#ff7f0e',  # orange
}
# fallback for any extra methods not in the dict above
extra_palette = plt.cm.tab10.colors
fallback_i = 0
for m in methods:
    if m not in base_colors:
        base_colors[m] = mcolors.to_hex(extra_palette[fallback_i])
        fallback_i += 1

algo_shade = {'Ward': 1.35, 'SKATER': 0.75}   # lighter for Ward, darker for SKATER
algo_markers = {'Ward': 'o', 'SKATER': 's'}
algo_display = {'Ward': 'SCHC', 'SKATER': 'SKATER'}   # legend-only rename

fig, axes = plt.subplots(3, 2, figsize=(11, 15), sharex=True, sharey=True)
for i, (ax, layout) in enumerate(zip(axes.flat, layouts)):
    d = grouped[grouped['layout'] == layout]
    for method in methods:
        m = d[d['method'] == method]
        for algo in ['Ward', 'SKATER']:
            a = m[m['algorithm'] == algo].sort_values('missing_bin_label')
            color = adjust_lightness(base_colors[method], algo_shade[algo])
            ax.plot(a['missing_bin_label'], a['ari'],
                     color=color, linestyle='-', linewidth=1.5,
                     marker=algo_markers[algo], markersize=3.5,
                     label=f'{method} – {algo_display[algo]}')
    ax.set_title(layout, fontsize=10)
    ax.tick_params(axis='both', labelsize=8)
    ax.tick_params(axis='x', rotation=90, labelsize=6)
    ax.grid(alpha=0.3)
    row, col = divmod(i, 2)
    ax.set_xlabel('% edges missing' if row == 2 else '', fontsize=9)
    ax.set_ylabel('Mean ARI' if col == 0 else '', fontsize=9)
    # fine grid on y only (major + minor)
    ax.yaxis.set_minor_locator(AutoMinorLocator(4))
    ax.grid(which='major', axis='y', alpha=0.3, linewidth=0.6)
    ax.grid(which='minor', axis='y', alpha=0.15, linewidth=0.4, linestyle=':')
    # x: gridline only at each category tick, no finer subdivision
    ax.grid(which='major', axis='x', alpha=0.3, linewidth=0.6)
    if col != 0:
        ax.tick_params(axis='y', labelleft=False)
    if i == 0:
        ax.legend(fontsize=6, loc='lower left')
    else:
        if ax.get_legend() is not None:
            ax.get_legend().remove()
#fig.suptitle('Mean ARI by method and algorithm across spatial layouts', fontsize=13, y=0.995)
fig.tight_layout(rect=[0.02, 0.01, 0.98, 0.965])
sns.despine()
plt.savefig("figs/reg_lineplot_by_method.png", dpi=600, bbox_inches="tight")

In [ ]:
subset = gdf.copy()

# discard degenerate borders/local rows where c_run == z_run
mask_drop = subset['method'].isin(['borders', 'local']) & (subset['c_run'] == subset['z_run'])
subset = subset[~mask_drop]

subset['layout'] = subset['shape'] + '_' + subset['size'].astype(str)
subset = subset.query("layout == 'hex_100'").copy()

# bin pct_missing into 2% wide bins, use bin midpoint for plotting
bins = np.arange(0, 100 + 2, 1)
subset['pct_bin'] = pd.cut(subset['pct_missing'], bins=bins)
subset['pct_bin_mid'] = subset['pct_bin'].apply(lambda x: x.mid if pd.notna(x) else np.nan)

grouped = subset.groupby(['method', 'pct_bin_mid'])[['ari_ward', 'ari_skater']].mean().reset_index()

methods = subset['method'].unique()
colors = plt.cm.tab10.colors

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)

for ax, algo_col, algo_label in zip(axes, ['ari_ward', 'ari_skater'], ['SCHC', 'SKATER']):
    for j, method in enumerate(methods):
        m = grouped[grouped['method'] == method].sort_values('pct_bin_mid')
        ax.plot(m['pct_bin_mid'], m[algo_col], color=colors[j], marker='o', markersize=3, label=method)
    ax.set_title(algo_label, fontsize=11)
    ax.set_xlabel('% edges missing', fontsize=9)
    ax.tick_params(axis='both', labelsize=8)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc='lower left')

axes[0].set_ylabel('Mean ARI', fontsize=9)
sns.despine()
#fig.suptitle('hex_100 – mean ARI by method', fontsize=13, y=1.0)
fig.tight_layout()
plt.savefig("figs/reg_method_compairson_binned1.png",dpi=600, bbox_inches="tight" )

# Zika

In [ ]:
zika = pd.read_parquet("summary/regionalization/real/metrics_ceara_zika.parquet")

In [ ]:
zika

In [ ]:
grouped = zika.groupby('missing_edges')[['ari_ward', 'ari_skater']].min().reset_index()
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(grouped['missing_edges'], grouped['ari_ward'], marker='o', label='Ward')
ax.plot(grouped['missing_edges'], grouped['ari_skater'], marker='o', label='SKATER')
ax.set_xlabel('Missing edge ratio')
ax.set_ylabel('ARI (mean)')
ax.set_title('ARI vs. missing edges')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
c_runs = sorted(zika['c_run'].unique())  # should be 0-9

fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharex=True, sharey=True)
axes = axes.flatten()

for i, run in enumerate(c_runs):
    ax = axes[i]
    sub = zika[zika['c_run'] == run]
    ax.plot(sub['missing_edges'], sub['ari_ward'], marker='o', label='Ward')
    ax.plot(sub['missing_edges'], sub['ari_skater'], marker='o', label='SKATER')
    ax.set_title(f'c_run {run}')
    ax.grid(alpha=0.3)

# shared labels instead of repeating on every subplot
fig.text(0.5, 0.02, 'Intensity', ha='center')
fig.text(0.08, 0.5, 'ARI', va='center', rotation='vertical')
axes[0].legend()

fig.suptitle('ARI vs. intensity per c_run (random removal)')
plt.tight_layout(rect=[0.03, 0.03, 1, 0.96])

In [ ]:
zika_extra = pd.read_parquet("summary/regionalization/real/extra/metrics_ceara_zika.parquet")

In [ ]:
c_runs = sorted(zika_extra['c_run'].unique())  # should be 0-9

fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharex=True, sharey=True)
axes = axes.flatten()
anchor = pd.DataFrame({'missing_edges': [0], 'ari_ward': [1], 'ari_skater': [1]})

for i, run in enumerate(c_runs):
    ax = axes[i]
    sub = zika_extra[zika_extra['c_run'] == run]
    ax.plot(sub['missing_edges'], sub['ari_ward'], marker='o', label='Ward')
    ax.plot(sub['missing_edges'], sub['ari_skater'], marker='o', label='SKATER')
    ax.set_title(f'c_run {run}')
    ax.grid(alpha=0.3)

# shared labels instead of repeating on every subplot
fig.text(0.5, 0.02, 'Intensity', ha='center')
fig.text(0.08, 0.5, 'ARI', va='center', rotation='vertical')
axes[0].legend()

fig.suptitle('ARI vs. intensity per c_run (random removal)')
plt.tight_layout(rect=[0.03, 0.03, 1, 0.96])

# Chicago

In [ ]:
chicago = pd.read_parquet("summary/regionalization/real/metrics_chicago.parquet")

In [ ]:
grouped = chicago.groupby('missing_edges')[['ami_ward', 'ami_skater']].mean().reset_index()
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(grouped['missing_edges'], grouped['ami_ward'], marker='o', label='Ward')
ax.plot(grouped['missing_edges'], grouped['ami_skater'], marker='o', label='SKATER')
ax.set_xlabel('Missing edge ratio')
ax.set_ylabel('ARI (mean)')
ax.set_title('ARI vs. missing edge ratio (random removal)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
c_runs = sorted(chicago['c_run'].unique())  # should be 0-9

fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharex=True, sharey=True)
axes = axes.flatten()

for i, run in enumerate(c_runs):
    ax = axes[i]
    sub = chicago[chicago['c_run'] == run]
    ax.plot(sub['missing_edges'], sub['ari_ward'], marker='o', label='Ward')
    ax.plot(sub['missing_edges'], sub['ari_skater'], marker='o', label='SKATER')
    ax.set_title(f'c_run {run}')
    ax.grid(alpha=0.3)

# shared labels instead of repeating on every subplot
fig.text(0.5, 0.02, 'Intensity', ha='center')
fig.text(0.08, 0.5, 'ARI', va='center', rotation='vertical')
axes[0].legend()

fig.suptitle('ARI vs. intensity per c_run (random removal)')
plt.tight_layout(rect=[0.03, 0.03, 1, 0.96])

In [ ]:
chicago_extra = pd.read_parquet("summary/regionalization/real/extra/metrics_chicago.parquet")

In [ ]:
runs = sorted(chicago_extra['c_run'].unique())

fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharex=True, sharey=True)
axes = axes.flatten()

anchor = pd.DataFrame({'missing_edges': [0], 'ari_ward': [1], 'ari_skater': [1]})

for ax, run in zip(axes, runs):
    sub = chicago_extra[chicago_extra['c_run'] == run].sort_values('missing_edges')
    sub = pd.concat([anchor, sub[['missing_edges', 'ari_ward', 'ari_skater']]], ignore_index=True)

    ax.plot(sub['missing_edges'], sub['ari_ward'], marker='o', label='Ward')
    ax.plot(sub['missing_edges'], sub['ari_skater'], marker='o', label='SKATER')
    ax.set_title(f'Run {run}')
    ax.grid(alpha=0.3)

for ax in axes[5:]:
    ax.set_xlabel('Missing edge ratio')
for ax in axes[::5]:
    ax.set_ylabel('ARI')

axes[0].legend()
fig.suptitle('ARI vs. missing edge ratio by c_run', y=1.02)
plt.tight_layout()
#plt.show()

# South

In [ ]:
south = pd.read_parquet("summary/regionalization/real/metrics_south.parquet")

In [ ]:
south

In [ ]:
grouped = south.groupby('missing_edges')[['ari_ward', 'ari_skater']].mean().reset_index()
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(grouped['missing_edges'], grouped['ari_ward'], marker='o', label='Ward')
ax.plot(grouped['missing_edges'], grouped['ari_skater'], marker='o', label='SKATER')
ax.set_xlabel('Missing edge ratio')
ax.set_ylabel('ARI (mean)')
ax.set_title('ARI vs. missing edge ratio (random removal)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
c_runs = sorted(south['c_run'].unique())  # should be 0-9

fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharex=True, sharey=True)
axes = axes.flatten()

for i, run in enumerate(c_runs):
    ax = axes[i]
    sub = south[south['c_run'] == run]
    ax.plot(sub['missing_edges'], sub['ami_ward'], marker='o', label='Ward')
    ax.plot(sub['missing_edges'], sub['ami_skater'], marker='o', label='SKATER')
    ax.set_title(f'c_run {run}')
    ax.grid(alpha=0.3)

# shared labels instead of repeating on every subplot
fig.text(0.5, 0.02, 'Intensity', ha='center')
fig.text(0.08, 0.5, 'ARI', va='center', rotation='vertical')
axes[0].legend()

fig.suptitle('ARI vs. intensity per c_run (random removal)')
plt.tight_layout(rect=[0.03, 0.03, 1, 0.96])

In [ ]:
def align_labels(true_labels, pred_labels):
    """
    Relabel pred_labels so that label IDs correspond as closely as
    possible to true_labels, using the Hungarian algorithm on the
    overlap (contingency) matrix. Unmatched/extra predicted clusters
    get new label IDs appended after the true ones, so nothing is lost.
    """
    true_labels = np.asarray(true_labels)
    pred_labels = np.asarray(pred_labels)

    true_ids = np.unique(true_labels)
    pred_ids = np.unique(pred_labels)

    # contingency[i, j] = number of units where true == true_ids[i] and pred == pred_ids[j]
    contingency = np.zeros((len(true_ids), len(pred_ids)), dtype=int)
    for i, t in enumerate(true_ids):
        for j, p in enumerate(pred_ids):
            contingency[i, j] = np.sum((true_labels == t) & (pred_labels == p))

    # maximize overlap -> minimize negative overlap
    row_ind, col_ind = linear_sum_assignment(-contingency)

    mapping = {}
    for i, j in zip(row_ind, col_ind):
        mapping[pred_ids[j]] = true_ids[i]

    # any predicted cluster not matched to a true cluster (more pred clusters than true)
    # gets a new unused label so it doesn't collide with a true label
    next_label = true_ids.max() + 1 if len(true_ids) else 0
    for p in pred_ids:
        if p not in mapping:
            mapping[p] = next_label
            next_label += 1

    return np.array([mapping[p] for p in pred_labels])

# --- prepare data ---
if 'obs_id' not in south_gdf.columns:
    south_gdf = south_gdf.reset_index(drop=True)
    south_gdf['obs_id'] = south_gdf.index

c_run0 = south_results[(south_results['method'] == 'random') & (south_results['c_run'] == 9)]
intensities = sorted(c_run0['intensity'].unique())

true_ward = south_gdf['true_skater'].values

fig, axes = plt.subplots(4, 5, figsize=(25, 20))
axes = axes.flatten()

# panel 0: ground truth (no alignment needed, it IS the reference)
south_gdf.plot(column='true_ward', categorical=True, cmap='tab20', ax=axes[0],
               edgecolor='white', linewidth=0.2, legend=False)
axes[0].set_title('True Ward', fontsize=12)
axes[0].axis('off')

# panels 1–19: relabeled to match true_ward as closely as possible
for i, intensity in enumerate(intensities, start=1):
    sub = c_run0[c_run0['intensity'] == intensity][['obs_id', 'ward_label']].sort_values('obs_id')
    merged = south_gdf[['obs_id', 'geometry']].merge(sub, on='obs_id', how='left')

    aligned = align_labels(true_ward, merged['ward_label'].values)
    merged['ward_label_aligned'] = aligned

    merged.plot(column='ward_label_aligned', categorical=True, cmap='tab20', ax=axes[i],
                edgecolor='white', linewidth=0.2, legend=False)
    axes[i].set_title(f'Intensity {intensity}', fontsize=12)
    axes[i].axis('off')

for j in range(len(intensities) + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Ward regionalization: true vs. corrupted (random removal, c_run = 0), labels aligned',
              fontsize=18)
plt.tight_layout()

In [ ]:
south_extra = pd.read_parquet("summary/regionalization/real/extra/metrics_south.parquet")

In [ ]:
grouped = south_extra.groupby('missing_edges')[['ari_ward', 'ari_skater']].min().reset_index()
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(grouped['missing_edges'], grouped['ari_ward'], marker='o', label='Ward')
ax.plot(grouped['missing_edges'], grouped['ari_skater'], marker='o', label='SKATER')
ax.set_xlabel('Missing edge ratio')
ax.set_ylabel('ARI (mean)')
ax.set_title('ARI vs. missing edge ratio (random removal)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
runs = sorted(south_extra['c_run'].unique())

fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharex=True, sharey=True)
axes = axes.flatten()

anchor = pd.DataFrame({'missing_edges': [0], 'ari_ward': [1], 'ari_skater': [1]})

for ax, run in zip(axes, runs):
    sub = south_extra[south_extra['c_run'] == run].sort_values('missing_edges')
    sub = pd.concat([anchor, sub[['missing_edges', 'ari_ward', 'ari_skater']]], ignore_index=True)

    ax.plot(sub['missing_edges'], sub['ari_ward'], marker='o', label='Ward')
    ax.plot(sub['missing_edges'], sub['ari_skater'], marker='o', label='SKATER')
    ax.set_title(f'Run {run}')
    ax.grid(alpha=0.3)

for ax in axes[5:]:
    ax.set_xlabel('Missing edge ratio')
for ax in axes[::5]:
    ax.set_ylabel('ARI')

axes[0].legend()
fig.suptitle('ARI vs. missing edge ratio by c_run', y=1.02)
plt.tight_layout()
plt.show()

# Real compairsons

In [ ]:
datasets = {
    'chicago': chicago,
    'ceara_zika': zika,
    'south': south,
}

processed = []
totals = {}  # dataset -> total undirected edge count

for name, df in datasets.items():
    g_true = graph.read_parquet(f"graphs/real/{name}/g_true.parquet")
    n_edges_total = g_true.n_edges / 2
    totals[name] = n_edges_total

    df = df.copy()
    df['dataset'] = name
    df['missing_edges_pct'] = df['missing_edges'] / n_edges_total * 100
    processed.append(df)
    
all_real = pd.concat(processed, ignore_index=True)

In [ ]:
all_real

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, name in zip(axes, datasets.keys()):
    sub = all_real[all_real['dataset'] == name]
    grouped = sub.groupby('missing_edges_pct')[['ari_ward', 'ari_skater']].mean().reset_index()

    ax.plot(grouped['missing_edges_pct'], grouped['ari_ward'], marker='o', label='Ward')
    ax.plot(grouped['missing_edges_pct'], grouped['ari_skater'], marker='o', label='SKATER')
    ax.set_xlabel('Missing edges (%)')
    ax.set_title(name.capitalize())
    ax.grid(alpha=0.3)

axes[0].set_ylabel('ARI (mean)')
axes[0].legend()
fig.suptitle('ARI vs. missing edge percentage (random removal)')
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 5), sharey=True)

bin_width = 5

for ax, name in zip(axes, datasets.keys()):
    sub = all_real[all_real['dataset'] == name].copy()

    # --- bin into 5% intervals ---
    max_pct = sub['missing_edges_pct'].max()
    bin_edges = np.arange(0, np.ceil(max_pct / bin_width) * bin_width + bin_width, bin_width)
    sub['missing_bin'] = pd.cut(sub['missing_edges_pct'], bins=bin_edges,
                                 right=False, include_lowest=True)
    sub['missing_bin_mid'] = sub['missing_bin'].apply(lambda x: x.mid if pd.notna(x) else np.nan)

    grouped = sub.groupby('missing_bin_mid')[['ari_ward', 'ari_skater']].mean().reset_index()
    grouped = grouped.rename(columns={'missing_bin_mid': 'missing_edges_pct'})

    # anchor point: 0% missing edges -> ARI = 1
    anchor = pd.DataFrame({'missing_edges_pct': [0], 'ari_ward': [1], 'ari_skater': [1]})
    grouped = pd.concat([anchor, grouped], ignore_index=True).sort_values('missing_edges_pct')

    ax.plot(grouped['missing_edges_pct'], grouped['ari_ward'], label='SCHC',
            marker='o', markersize=4, linewidth=1.5)
    ax.plot(grouped['missing_edges_pct'], grouped['ari_skater'], label='SKATER',
            marker='s', markersize=4, linewidth=1.5)

    ax.set_xlabel('% edges missing')
    ax.set_title(name.capitalize())

    # --- finer gridlines ---
    ax.yaxis.set_major_locator(MultipleLocator(0.1))
    ax.yaxis.set_minor_locator(MultipleLocator(0.025))
    ax.grid(which='major', axis='y', alpha=0.35, linewidth=0.6)
    ax.grid(which='minor', axis='y', alpha=0.15, linewidth=0.4, linestyle=':')
    ax.grid(which='major', axis='x', alpha=0.2, linewidth=0.6)

    # --- interval-style xticks, positioned at bin midpoints ---
    bin_starts = bin_edges[:-1]
    tick_positions = bin_starts + bin_width / 2
    tick_labels = [f"{int(s)}-{int(s + bin_width)}" for s in bin_starts]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=90, fontsize=7)

axes[0].legend(loc='upper right', fontsize=8)
sns.despine()
axes[0].set_ylabel('ARI (mean)')
plt.tight_layout()
plt.savefig("figs/reg_real_all.png", dpi=600, bbox_inches="tight")

In [ ]:
datasets = {
    'chicago': chicago_extra,
    'ceara_zika': zika_extra,
    'south': south_extra,
}

processed = []
totals = {}  # dataset -> total undirected edge count

for name, df in datasets.items():
    g_true = graph.read_parquet(f"graphs/real/{name}/g_true.parquet")
    n_edges_total = g_true.n_edges / 2
    totals[name] = n_edges_total

    df = df.copy()
    df['dataset'] = name
    df['missing_edges_pct'] = df['missing_edges'] / n_edges_total * 100
    processed.append(df)
    
all_extra = pd.concat(processed, ignore_index=True)

In [ ]:
selections = {
    'chicago': 5,
    'ceara_zika': 9,
    'south': 1,
}

fig, axes = plt.subplots(1, 3, figsize=(11, 5), sharey=True)

for ax, (name, run) in zip(axes, selections.items()):
    sub = all_extra[(all_extra['dataset'] == name) & (all_extra['c_run'] == run)]
    grouped = sub.groupby('missing_edges_pct')[['ari_ward', 'ari_skater']].mean().reset_index()

    # anchor point: 0% missing edges -> ARI = 1
    anchor = pd.DataFrame({'missing_edges_pct': [0], 'ari_ward': [1], 'ari_skater': [1]})
    grouped = pd.concat([anchor, grouped], ignore_index=True).sort_values('missing_edges_pct')

    base = name.replace('_extra', '')
    total = totals[base]

    # --- convert primary x-axis to edge COUNT, keep % as secondary (top) axis ---
    grouped['missing_edges_count'] = grouped['missing_edges_pct'] / 100 * total

    ax.plot(grouped['missing_edges_count'], grouped['ari_ward'], label='SCHC', linewidth=1.5)
    ax.plot(grouped['missing_edges_count'], grouped['ari_skater'], label='SKATER', linewidth=1.5)

    ax.set_xlabel('Missing edges (count)')
    ax.set_title(f"{base.capitalize()}")

    # --- finer gridlines ---
    ax.yaxis.set_major_locator(MultipleLocator(0.1))
    ax.yaxis.set_minor_locator(MultipleLocator(0.025))
    ax.grid(which='major', axis='y', alpha=0.35, linewidth=0.6)
    ax.grid(which='minor', axis='y', alpha=0.15, linewidth=0.4, linestyle=':')
    ax.grid(which='major', axis='x', alpha=0.2, linewidth=0.6)
    ax.xaxis.set_minor_locator(AutoMinorLocator(4))
    ax.grid(which='minor', axis='x', alpha=0.1, linewidth=0.4, linestyle=':')

    secax = ax.secondary_xaxis(
        'top',
        functions=(lambda x, t=total: x / t * 100,
                   lambda x, t=total: x / 100 * t)
    )
    secax.set_xlabel('Missing edges (%)')

    if name == 'south':
        ax.legend(loc='lower right')

sns.despine()
axes[0].set_ylabel('ARI')
plt.tight_layout()
plt.savefig("figs/reg_real_examples.png", dpi=600, bbox_inches="tight")

In [ ]:
all_extra["ari_ward"].min()

In [ ]:
for name, run in selections.items():
    sub = all_extra[(all_extra['dataset'] == name) & (all_extra['c_run'] == run)]
    print(name, 'ami_skater min (this run):', sub['ami_skater'].min())
    print(name, 'ari_skater min (this run):', sub['ari_skater'].min())
    print(name, 'ami_skater min (all runs):', all_extra[all_extra['dataset']==name]['ami_skater'].min())

In [ ]:
all_extra.query("dataset=='ceara_zika' and c_run == 9")

In [ ]:
pd.read_parquet("results/regionalization/real/extra/reg_south.parquet")

In [ ]:
gpd.read_parquet("data/real/south.parquet")

In [ ]:

def _align_labels(true_labels, pred_labels):
    """Relabel pred_labels so each predicted cluster takes the id of its
    best-overlapping true cluster (max overlap matching via Hungarian algorithm)."""
    true_ids = np.sort(true_labels.dropna().unique())
    pred_ids = np.sort(pred_labels.dropna().unique())

    contingency = pd.crosstab(pred_labels, true_labels).reindex(
        index=pred_ids, columns=true_ids, fill_value=0
    )
    row_ind, col_ind = linear_sum_assignment(-contingency.values)  # maximize overlap

    mapping = {pred_ids[r]: true_ids[c] for r, c in zip(row_ind, col_ind)}
    next_id = int(max(true_ids.max(), pred_ids.max())) + 1
    for pid in pred_ids:
        if pid not in mapping:
            mapping[pid] = next_id
            next_id += 1

    return pred_labels.map(mapping)


def plot_missing_edges_map(dataset, n_missing, c_run, algorithm,
                            reg_dir="results/regionalization/real/extra",
                            geo_dir="data/real",
                            all_extra=all_extra,
                            palette='Set2'):
    assert algorithm in ('ward', 'skater'), "algorithm must be 'ward' or 'skater'"
    display_names = {'ward': 'SCHC', 'skater': 'SKATER'}
    display_name = display_names[algorithm]
    gdf = gpd.read_parquet(f"{geo_dir}/{dataset}.parquet")
    reg = pd.read_parquet(f"{reg_dir}/reg_{dataset}.parquet")
    sub = reg[(reg['c_run'] == c_run) & (reg['missing_edges'] == n_missing)].set_index('obs_id')
    merged = gdf.copy()
    merged['pred_label'] = sub[f'{algorithm}_label']
    ari_row = all_extra[
        (all_extra['dataset'] == dataset) &
        (all_extra['c_run'] == c_run) &
        (all_extra['missing_edges'] == n_missing)
    ]
    ari = ari_row[f'ari_{algorithm}'].values[0]
    true_col = f'true_{algorithm}'
    merged['pred_label_aligned'] = _align_labels(merged[true_col], merged['pred_label'])

    # shared color mapping across both panels
    n_colors = int(max(merged[true_col].max(), merged['pred_label_aligned'].max())) + 1
    base_colors = plt.get_cmap(palette).colors
    cmap = ListedColormap([base_colors[i % len(base_colors)] for i in range(n_colors)])

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    merged.plot(column=true_col, ax=axes[0], cmap=cmap, vmin=0, vmax=n_colors - 1,
                edgecolor='white', linewidth=0.3,
                missing_kwds={'color': 'lightgrey'})
    axes[0].set_title("Ground truth")
    axes[0].axis('off')

    merged.plot(column='pred_label_aligned', ax=axes[1], cmap=cmap, vmin=0, vmax=n_colors - 1,
                edgecolor='white', linewidth=0.2,
                missing_kwds={'color': 'lightgrey'})
    axes[1].set_title(f"{display_name} ({n_missing} missing edges, ARI={ari:.3f})")
    axes[1].axis('off')

    plt.tight_layout()
    plt.savefig(f"figs/reg_{dataset}_map.png", dpi=600, bbox_inches="tight")

In [ ]:
plot_missing_edges_map('ceara_zika', 16, c_run=9,algorithm='ward', palette='Pastel1')

In [ ]:
plot_missing_edges_map('chicago', 2, c_run=5, algorithm='ward', palette='Pastel1')

In [ ]:
plot_missing_edges_map('south', 1, c_run=1, algorithm='skater', palette='Pastel1')

# South methods

In [ ]:
south_extra = pd.read_parquet("summary/regionalization/real/extra/metrics_south_methods.parquet")

In [ ]:
south_extra["missing_edges"].max()

In [ ]:
g_true = graph.read_parquet(f"graphs/real/south/g_true.parquet")
n_edges_total = g_true.n_edges / 2
south_extra = south_extra.copy()

south_extra['missing_edges_pct'] = south_extra['missing_edges'] / n_edges_total * 100

In [ ]:
# --- bin into 5% intervals ---
import matplotlib.colors as mcolors
import colorsys
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator

bin_width = 1
max_pct = south_extra['missing_edges_pct'].max()
bin_edges = np.arange(0, np.ceil(max_pct / bin_width) * bin_width + bin_width, bin_width)
south_extra['missing_bin'] = pd.cut(
    south_extra['missing_edges_pct'], bins=bin_edges,
    right=False, include_lowest=True
)

def fmt_bin(iv):
    return f"{iv.left:g}-{iv.right:g}"

south_extra['missing_bin_label'] = south_extra['missing_bin'].map(fmt_bin)
bin_order = [fmt_bin(iv) for iv in south_extra['missing_bin'].cat.categories]

long_df = south_extra.melt(
    id_vars=['method', 'missing_bin_label'],
    value_vars=['ari_ward', 'ari_skater'],
    var_name='algorithm', value_name='ari'
)
long_df['algorithm'] = long_df['algorithm'].map({'ari_ward': 'Ward', 'ari_skater': 'SKATER'})
long_df['missing_bin_label'] = pd.Categorical(
    long_df['missing_bin_label'], categories=bin_order, ordered=True
)

grouped = (
    long_df.groupby(['method', 'algorithm', 'missing_bin_label'], observed=True)['ari']
    .mean()
    .reset_index()
)

methods = south_extra['method'].unique()

# --- base color per method, shaded by algorithm ---
def adjust_lightness(hex_color, factor):
    r, g, b = mcolors.to_rgb(hex_color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = max(0, min(1, l * factor))
    return colorsys.hls_to_rgb(h, l, s)

base_colors = {
    'random':  '#1f77b4',  # blue
    'borders': '#2ca02c',  # green
    'local':   '#ff7f0e',  # orange
}
extra_palette = plt.cm.tab10.colors
fallback_i = 0
for m in methods:
    if m not in base_colors:
        base_colors[m] = mcolors.to_hex(extra_palette[fallback_i])
        fallback_i += 1

algo_shade = {'Ward': 1.35, 'SKATER': 0.75}
algo_markers = {'Ward': 'o', 'SKATER': 's'}
algo_display = {'Ward': 'SCHC', 'SKATER': 'SKATER'}  # only for the legend text


fig, ax = plt.subplots(figsize=(8, 6))

for method in methods:
    m = grouped[grouped['method'] == method]
    for algo in ['Ward', 'SKATER']:
        a = m[m['algorithm'] == algo].sort_values('missing_bin_label')
        color = adjust_lightness(base_colors[method], algo_shade[algo])
        ax.plot(a['missing_bin_label'], a['ari'],
                color=color, linestyle='-', linewidth=1.5,
                marker=algo_markers[algo], markersize=4,
                label=f'{method} – {algo_display[algo]}')
        
ax.set_xlabel('% edges missing')
ax.set_ylabel('Mean ARI')
#ax.set_title('ARI vs Missing Edges % (binned every 5%), by method')
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.yaxis.set_minor_locator(AutoMinorLocator(4))
ax.grid(which='major', axis='y', alpha=0.3, linewidth=0.6)
ax.grid(which='minor', axis='y', alpha=0.15, linewidth=0.4, linestyle=':')
ax.grid(which='major', axis='x', alpha=0.3, linewidth=0.6)
ax.legend(fontsize=8, loc='lower left')
sns.despine()
plt.tight_layout()
plt.savefig("figs/south_extra_lineplot_by_method.png", dpi=600, bbox_inches="tight")
#plt.show()

In [ ]:
south_extra

In [ ]:
south = gpd.read_parquet("data/real/south.parquet")

In [ ]:
south

In [ ]:
g = graph.read_parquet("graphs/real/south/one_component/local/g_missing15_run51.parquet")

In [ ]:
tools.plot_graph(south,g)

In [ ]:
south_extra

In [ ]:
import glob, re

# what files actually exist for local method
local_files = sorted(glob.glob("graphs/real/south/one_component/local/*.parquet"))
print(local_files[:20])

# what run/missing values does the code actually expect vs produce
pattern = re.compile(r"g_missing(\d+)_run(\d+)\.parquet")
parsed = [pattern.match(f.split("/")[-1]) for f in local_files]
seen_runs = sorted(set(int(m.group(2)) for m in parsed if m))
print("actual c_run values found on disk:", seen_runs)